In [ ]:
!pip install feedparser

import feedparser
import pandas as pd
from datetime import datetime, timezone

In [ ]:
feeds = {
    "TechCrunch Venture": "https://techcrunch.com/category/venture/feed/",
    "TechCrunch Startups": "https://techcrunch.com/category/startups/feed/",
    "VentureBeat": "https://venturebeat.com/feed/",
}

raw_entries = []

for source, url in feeds.items():
    feed = feedparser.parse(url)
    for entry in feed.entries:
        raw_entries.append({
            "source": source,
            "title": entry.get("title", ""),
            "link": entry.get("link", ""),
            "summary": entry.get("summary", ""),
            "published": entry.get("published", ""),
        })

print(f"Total entries fetched: {len(raw_entries)}")

Total entries fetched: 47


In [ ]:
funding_keywords = [
    "raises", "raised", "funding", "seed", "series a", "series b",
    "investment", "backed", "venture", "million", "capital"
]

sector_keywords = [
    # AI / ML
    "ai", "artificial intelligence", "machine learning", "llm", "generative",
    # B2B SaaS
    "saas", "b2b", "enterprise", "platform", "workflow", "automation",
    # Marketplace / consumer
    "marketplace", "consumer", "app", "platform", "fintech", "edtech", "healthtech"
]

def is_relevant(entry):
    text = (entry["title"] + " " + entry["summary"]).lower()
    has_funding = any(kw in text for kw in funding_keywords)
    has_sector = any(kw in text for kw in sector_keywords)
    return has_funding and has_sector

filtered = [e for e in raw_entries if is_relevant(e)]
df = pd.DataFrame(filtered)

print(f"Relevant entries after filter: {len(filtered)}")
df[["source", "title", "published", "link"]]

Relevant entries after filter: 31


,source,title,published,link
0,TechCrunch Venture,Convective Capital raises an $85 million fund ...,"Thu, 21 May 2026 17:41:20 +0000",https://techcrunch.com/2026/05/21/convective-c...
1,TechCrunch Venture,This young startup is taking on a fragrance in...,"Thu, 21 May 2026 16:00:00 +0000",https://techcrunch.com/2026/05/21/a-new-fragra...
2,TechCrunch Venture,Beauty booking startup Fresha hits $1B valuati...,"Thu, 21 May 2026 11:00:00 +0000",https://techcrunch.com/2026/05/21/booking-plat...
3,TechCrunch Venture,Imperagen raises £5 million to use quantum phy...,"Thu, 21 May 2026 04:00:00 +0000",https://techcrunch.com/2026/05/20/imperagen-ra...
4,TechCrunch Venture,How Lucra raised $20M as an eSports play when ...,"Wed, 20 May 2026 14:30:00 +0000",https://techcrunch.com/podcast/how-lucra-raise...
5,TechCrunch Venture,Forget the feed: Status AI raises $17M to turn...,"Tue, 19 May 2026 14:00:00 +0000",https://techcrunch.com/2026/05/19/gamified-soc...
6,TechCrunch Venture,Stilta raises $10.5M from a16z and YC to help ...,"Tue, 19 May 2026 12:00:00 +0000",https://techcrunch.com/2026/05/19/legal-tech-a...
7,TechCrunch Venture,Marketing operating system Nectar Social raise...,"Sat, 16 May 2026 19:26:14 +0000",https://techcrunch.com/2026/05/16/marketing-op...
8,TechCrunch Venture,$60B AI chip darling Cerebras almost died earl...,"Sat, 16 May 2026 15:00:00 +0000",https://techcrunch.com/2026/05/16/60b-ai-chip-...
9,TechCrunch Venture,Meridian Ventures launched a $35M fund with a ...,"Fri, 15 May 2026 13:00:00 +0000",https://techcrunch.com/2026/05/15/meridian-ven...


In [ ]:
from datetime import datetime, timezone, timedelta
import email.utils

# Step 1: Fresh start from raw_entries
df = pd.DataFrame(raw_entries)

# Step 2: Deduplicate on title
df = df.drop_duplicates(subset="title")

# Step 3: Parse dates and apply 7-day filter
def parse_date(date_str):
    try:
        return email.utils.parsedate_to_datetime(date_str).replace(tzinfo=timezone.utc)
    except:
        return None

df["parsed_date"] = df["published"].apply(parse_date)
cutoff = datetime.now(timezone.utc) - timedelta(days=7)
df = df[df["parsed_date"].notna() & (df["parsed_date"] >= cutoff)]

# Step 4: Must contain a funding signal in the TITLE only (not summary)
# Title-only matching is much more precise
funding_title_keywords = [
    "raises", "raised", "funding", "seed", "series a", "series b",
    "backed", "million", "launches", "announces"
]

def has_funding_signal(title):
    t = title.lower()
    return any(kw in t for kw in funding_title_keywords)

df = df[df["title"].apply(has_funding_signal)]

# Step 5: Exclude noise by exact title phrases
exclude_title_phrases = [
    "applications close",
    "third fund",
    "closed a fund",
    "retail venture ipo",
    "doubles valuation",
    "disrupt 2026",
    "50% off",
    "get ready for",
    "hottest place",
    "nvidia has",
    "google's new",
    "google says",
    "google just",
    "google unveils",
    "openai co-founder",
    "anthropic warns",
    "apple unveils",
    "stage at techcrunch",
]

def not_noise(title):
    t = title.lower()
    return not any(phrase in t for phrase in exclude_title_phrases)

df = df[df["title"].apply(not_noise)]
df = df.reset_index(drop=True)

print(f"Clean signal count: {len(df)}")
df[["source", "title", "published", "link"]]

Clean signal count: 9


,source,title,published,link
0,TechCrunch Venture,Convective Capital raises an $85 million fund ...,"Thu, 21 May 2026 17:41:20 +0000",https://techcrunch.com/2026/05/21/convective-c...
1,TechCrunch Venture,Imperagen raises £5 million to use quantum phy...,"Thu, 21 May 2026 04:00:00 +0000",https://techcrunch.com/2026/05/20/imperagen-ra...
2,TechCrunch Venture,How Lucra raised $20M as an eSports play when ...,"Wed, 20 May 2026 14:30:00 +0000",https://techcrunch.com/podcast/how-lucra-raise...
3,TechCrunch Venture,Forget the feed: Status AI raises $17M to turn...,"Tue, 19 May 2026 14:00:00 +0000",https://techcrunch.com/2026/05/19/gamified-soc...
4,TechCrunch Venture,Stilta raises $10.5M from a16z and YC to help ...,"Tue, 19 May 2026 12:00:00 +0000",https://techcrunch.com/2026/05/19/legal-tech-a...
5,TechCrunch Startups,"NanoClaw creator turns down $20M buyout offer,...","Wed, 20 May 2026 14:00:00 +0000",https://techcrunch.com/2026/05/20/nanoclaw-cre...
6,TechCrunch Startups,This startup raised $43M to build a hive mind ...,"Wed, 20 May 2026 13:00:00 +0000",https://techcrunch.com/2026/05/20/this-startup...
7,TechCrunch Startups,"From teen hacker to Iron Dome researcher, this...","Tue, 19 May 2026 21:08:51 +0000",https://techcrunch.com/2026/05/19/from-teen-ha...
8,VentureBeat,D&B's database of 642 million businesses was b...,"Fri, 22 May 2026 13:00:00 GMT",https://venturebeat.com/data/d-and-bs-database...


In [ ]:
!pip install anthropic

  Using cached pydantic_core-2.33.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached pydantic_core-2.33.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.0 MB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 1.15.1 requires rich<15.0.0,>=13.7.0, but you have rich 15.0.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.

In [ ]:
!pip install groq

from groq import Groq
from google.colab import userdata

In [ ]:
# Add a new secret in Colab called GROQ_API_KEY
# Get your free key at console.groq.com — no credit card needed
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def extract_startup_info(title):
    prompt = f"""
Extract startup funding information from this headline.
Return ONLY a JSON object with these exact keys:
- company: startup company name (string)
- amount: funding amount with $ sign (string, or "unknown" if not mentioned)
- stage: funding stage like Seed, Series A, Series B (string, or "unknown")
- sector: one of AI, SaaS, LegalTech, FinTech, Consumer, Hardware, Other (string)
- is_startup: true if this is about a startup raise, false if it's a VC fund or large corp (boolean)

Headline: "{title}"

Return only the JSON, no explanation, no markdown.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )

    try:
        return json.loads(response.choices[0].message.content)
    except:
        return {"company": "parse error", "amount": "unknown",
                "stage": "unknown", "sector": "unknown", "is_startup": False}

In [ ]:
print(len(df))
print(df.head())

9
               source                                              title  \
0  TechCrunch Venture  Convective Capital raises an $85 million fund ...   
1  TechCrunch Venture  Imperagen raises £5 million to use quantum phy...   
2  TechCrunch Venture  How Lucra raised $20M as an eSports play when ...   
3  TechCrunch Venture  Forget the feed: Status AI raises $17M to turn...   
4  TechCrunch Venture  Stilta raises $10.5M from a16z and YC to help ...   

                                                link  \
0  https://techcrunch.com/2026/05/21/convective-c...   
1  https://techcrunch.com/2026/05/20/imperagen-ra...   
2  https://techcrunch.com/podcast/how-lucra-raise...   
3  https://techcrunch.com/2026/05/19/gamified-soc...   
4  https://techcrunch.com/2026/05/19/legal-tech-a...   

                                             summary  \
0  After launching to invest in fire tech, Convec...   
1  Biotech company Imperagen announced on Thursda...   
2  Slapping &#8220;AI&#8221; on your

In [ ]:
!pip install groq -q

import json
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def extract_startup_info(title, summary):
    prompt = f"""
Extract startup funding information from this headline and summary.
Return ONLY a JSON object with these exact keys:
- company: startup company name (string)
- amount: funding amount with $ sign (string, or "unknown" if not mentioned)
- stage: funding stage like Seed, Series A, Series B (string, or "unknown")
- sector: one of AI, SaaS, LegalTech, FinTech, Consumer, Hardware, Other (string)
- is_startup: true if this is about a startup raise, false if it's a VC fund or large corp (boolean)

Headline: "{title}"
Summary: "{summary}"

Return only the JSON, no explanation, no markdown.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )

    try:
        return json.loads(response.choices[0].message.content)
    except:
        return {"company": "parse error", "amount": "unknown",
                "stage": "unknown", "sector": "unknown", "is_startup": False}

# Run extraction on all rows
extracted = []
for _, row in df.iterrows():
    info = extract_startup_info(row["title"], row["summary"])
    info["link"] = row["link"]
    info["published"] = row["published"]
    extracted.append(info)

df_structured = pd.DataFrame(extracted)

# Drop anything flagged as not a startup
df_structured = df_structured[df_structured["is_startup"] == True]
df_structured = df_structured.reset_index(drop=True)

print(f"Actionable startup targets: {len(df_structured)}")
df_structured[["company", "amount", "stage", "sector", "published", "link"]]

Actionable startup targets: 7


,company,amount,stage,sector,published,link
0,Imperagen,$6.7 million,Seed,Other,"Thu, 21 May 2026 04:00:00 +0000",https://techcrunch.com/2026/05/20/imperagen-ra...
1,Lucra,$20M,unknown,Other,"Wed, 20 May 2026 14:30:00 +0000",https://techcrunch.com/podcast/how-lucra-raise...
2,Status AI,$17M,unknown,AI,"Tue, 19 May 2026 14:00:00 +0000",https://techcrunch.com/2026/05/19/gamified-soc...
3,Stilta,$10.5M,unknown,AI,"Tue, 19 May 2026 12:00:00 +0000",https://techcrunch.com/2026/05/19/legal-tech-a...
4,NanoClaw,$12M,Seed,AI,"Wed, 20 May 2026 14:00:00 +0000",https://techcrunch.com/2026/05/20/nanoclaw-cre...
5,unknown,$43M,unknown,Other,"Wed, 20 May 2026 13:00:00 +0000",https://techcrunch.com/2026/05/20/this-startup...
6,Ocean,$28M,unknown,AI,"Tue, 19 May 2026 21:08:51 +0000",https://techcrunch.com/2026/05/19/from-teen-ha...


In [ ]:
def score_fit(company, amount, stage, sector):
    prompt = f"""
You are evaluating whether a recently funded startup is a good target for a PM job outreach.

Candidate background:
- 7+ years experience across Amazon, Deloitte, TCS
- Strong in B2B SaaS, marketplace, AI-powered products
- MBA from UC Davis (graduating 2026)
- Looking for Product Manager roles at early to mid stage startups
- Prefers Series A and Series B but open to well-funded Seed
- Less interested in Hardware and pure consumer social

Startup details:
- Company: {company}
- Funding: {amount} ({stage})
- Sector: {sector}

Return ONLY a JSON object with these exact keys:
- fit_score: integer from 1 to 10 (10 = perfect fit)
- reason: one sentence explaining the score
- action: one of "reach out now", "monitor", "skip"

Return only the JSON, no explanation, no markdown.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )

    try:
        return json.loads(response.choices[0].message.content)
    except:
        return {"fit_score": 0, "reason": "parse error", "action": "skip"}

# Run fit scoring on structured results
for _, row in df_structured.iterrows():
    result = score_fit(row["company"], row["amount"], row["stage"], row["sector"])
    df_structured.loc[_, "fit_score"] = result["fit_score"]
    df_structured.loc[_, "reason"] = result["reason"]
    df_structured.loc[_, "action"] = result["action"]

# Sort by fit score
df_structured = df_structured.sort_values("fit_score", ascending=False).reset_index(drop=True)

print("Ranked startup targets:")
df_structured[["company", "amount", "stage", "sector", "fit_score", "action", "reason"]]

Ranked startup targets:


,company,amount,stage,sector,fit_score,action,reason
0,Status AI,$17M,unknown,AI,8.0,monitor,The startup's focus on AI aligns with the cand...
1,NanoClaw,$12M,Seed,AI,8.0,reach out now,The startup's AI sector aligns with the candid...
2,Stilta,$10.5M,unknown,AI,8.0,monitor,The startup's focus on AI aligns with the cand...
3,Ocean,$28M,unknown,AI,8.0,reach out now,The startup's focus on AI aligns with the cand...
4,Imperagen,$6.7 million,Seed,Other,6.0,monitor,The startup's seed funding and unclear sector ...
5,Lucra,$20M,unknown,Other,4.0,monitor,The startup's funding and sector do not clearl...
6,unknown,$43M,unknown,Other,4.0,monitor,"The startup's funding and sector are unknown, ..."


In [ ]:
#CrewAI agents with defined roles
#Post 3: "Turned the script into a multi-agent system with CrewAI"
#Post 4: "Agents now log targets to Google Sheets automatically"
#Step 1: Install CrewAI in Colab
# Step 2: Define 3 agents — Scout, Researcher, Analyst
# Step 3: Define tasks for each agent
# Step 4: Wire agents into a Crew and run it
# Step 5: Output is the same ranked df you already have

In [ ]:
# ============================================
# Startup Signal Tracker — CrewAI Pipeline
# Phase 1: RSS Fetch → Extract → Score
# ============================================
#!pip install -q crewai==1.14.5 crewai-tools==1.14.5 groq==1.2.0 feedparser==6.0.12 gspread==6.2.1 google-auth==2.53.0 litellm==1.85.0

In [ ]:
# Cell 1 — reinstall with litellm explicit
!pip install -q crewai groq feedparser gspread google-auth litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 10.0 MB/s eta 0:00:00


In [ ]:
# Cell 2 — check versions
import pydantic, crewai
print("pydantic:", pydantic.__version__)
print("crewai:", crewai.__version__)

pydantic: 2.12.3
crewai: 1.14.5


In [ ]:
# Cell 3 — disable cache and set up LLM
import os
os.environ["LITELLM_CACHE"] = "False"
os.environ["LITELLM_ENABLE_CACHING"] = "False"
os.environ["GROQ_CACHE"] = "False"

import litellm
litellm.cache = None
litellm.caching = False

from crewai import Agent, Task, Crew, Process, LLM
from google.colab import userdata

llm = LLM(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)
print("LLM ready ✅")

20:50:22 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
20:50:22 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


LLM ready ✅


In [ ]:
# Cell 3
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(
    model="llama-3.3-70b-versatile",
    api_key=os.environ.get("GROQ_API_KEY", ""),
    base_url="https://api.groq.com/openai/v1"
)
print("LLM ready ✅")

LLM ready ✅


In [ ]:
# Step 2b: Define the 3 agents
from google.colab import userdata

scout = Agent(
    role="Startup Signal Scout",
    goal="Fetch RSS feeds from tech news sources, filter for recently funded startups, and return a clean deduplicated list of relevant entries from the last 7 days.",
    backstory="""You are an expert at monitoring startup news. You know how to spot
    genuine funding announcements and filter out noise, duplicates, and irrelevant content.
    You focus only on startups — not acquisitions, not public companies.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

researcher = Agent(
    role="Startup Data Researcher",
    goal="Extract structured data from startup funding entries: company name, funding amount, stage, and sector.",
    backstory="""You are a precise data extractor. Given raw RSS titles and summaries,
    you identify and pull out key funding details. You return clean, structured information
    and use 'Unknown' when data is not available rather than guessing.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

analyst = Agent(
    role="PM Fit Analyst",
    goal="Score each startup 1-10 for PM fit based on a candidate background, and recommend an action: reach out now, monitor, or skip.",
    backstory="""You evaluate startups as potential employers for a senior PM candidate.
    The candidate has 7+ years across Amazon, Deloitte, and TCS, strong B2B SaaS and AI product
    experience, an MBA from UC Davis (2026), and is targeting Series A/B startups.
    They prefer AI, SaaS, and marketplace companies. Hardware and pure consumer social are low fit.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Agents defined: Scout, Researcher, Analyst")

✅ Agents defined: Scout, Researcher, Analyst


In [ ]:
# Step 3: Define Tasks for each agent
from crewai import Task

# --- Task 1: Scout fetches and filters RSS entries ---
scout_task = Task(
    description="""
    Fetch RSS feeds from these 3 URLs:
    - https://techcrunch.com/feed/
    - https://techcrunch.com/category/startups/feed/
    - https://venturebeat.com/feed/

    Then:
    1. Filter entries where the title contains funding keywords:
       'raises', 'funding', 'million', 'series', 'seed', 'venture', 'backed', 'investment'
    2. Filter entries published in the last 7 days only
    3. Deduplicate on title
    4. Exclude entries with noise phrases: 'layoffs', 'acqui', 'IPO', 'public offering', 'bankrupt'
    5. Return a list of entries, each with: title, summary, published date, and link

    Use the feedparser library to fetch the feeds.
    """,
    expected_output="""
    A list of dictionaries, each containing:
    - title: string
    - summary: string
    - published: string
    - link: string

    Example:
    [
      {
        "title": "Status AI raises $17M Series A",
        "summary": "Status AI, an enterprise communication platform...",
        "published": "2025-05-15",
        "link": "https://techcrunch.com/..."
      }
    ]
    """,
    agent=scout
)

# --- Task 2: Researcher extracts structured data ---
researcher_task = Task(
    description="""
    You will receive a list of startup funding entries from the Scout.

    For each entry, extract:
    - company: the startup's name
    - amount: funding amount (e.g. '$17M', '$30M') or 'Unknown'
    - stage: funding stage (e.g. 'Seed', 'Series A', 'Series B') or 'Unknown'
    - sector: industry/sector (e.g. 'AI', 'FinTech', 'SaaS', 'LegalTech') or 'Unknown'
    - is_startup: true or false (exclude if public company or large enterprise)

    Use the title and summary to extract this information.
    Return only entries where is_startup is true.
    """,
    expected_output="""
    A list of dictionaries, each containing:
    - company: string
    - amount: string
    - stage: string
    - sector: string
    - title: string
    - link: string

    Example:
    [
      {
        "company": "Status AI",
        "amount": "$17M",
        "stage": "Series A",
        "sector": "AI",
        "title": "Status AI raises $17M Series A",
        "link": "https://techcrunch.com/..."
      }
    ]
    """,
    agent=researcher,
    context=[scout_task]
)

# --- Task 3: Analyst scores PM fit ---
# --- Task 3: Analyst scores PM fit ---
analyst_task = Task(
    description="""
    You will receive a list of structured startup entries from the Researcher.

    Score each startup for PM fit on a scale of 1-10 based on this candidate profile:
    - 7+ years experience across Amazon, Deloitte, TCS
    - Strong in B2B SaaS, marketplace, and AI-powered products
    - MBA from UC Davis, graduating 2026
    - Targeting Series A and Series B startups
    - Open to well-funded Seed rounds
    - Low interest in Hardware and pure consumer social

    For each startup provide:
    - company: copy the EXACT company name from Researcher output (never use 'Unknown' or placeholder names)
    - amount: copy exactly from Researcher
    - stage: copy exactly from Researcher
    - sector: copy exactly from Researcher
    - fit_score: integer 1-10
    - reason: one sentence explaining the score
    - action: one of 'reach out now' (8-10), 'monitor' (5-7), 'skip' (1-4)
    - key_people: founder/CEO/CTO names if mentioned in title or summary, otherwise 'Not found'

    Return ONLY a valid JSON array. No explanation, no markdown, no code blocks.
    Sort by fit_score descending.
    Example:
    [{"company": "Status AI", "amount": "$17M", "stage": "Series A", "sector": "AI", "fit_score": 9, "reason": "Strong AI B2B fit", "action": "reach out now", "key_people": "Jane Smith (CEO)"}]
    """,
    expected_output="Valid JSON array sorted by fit_score descending",
    agent=analyst,
    context=[researcher_task]
)

print("✅ Tasks defined: scout_task, researcher_task, analyst_task")

✅ Tasks defined: scout_task, researcher_task, analyst_task


In [ ]:
# Run this BEFORE the crew kickoff cell
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["OPENAI_MODEL_NAME"] = "llama-3.3-70b-versatile"

print("API config set ✅")

API config set ✅


In [ ]:
# Step 4: Create and run the Crew
import asyncio
import threading
from crewai import Crew, Process

crew = Crew(
    agents=[scout, researcher, analyst],
    tasks=[scout_task, researcher_task, analyst_task],
    process=Process.sequential,
    verbose=True
)

# Run crew in a separate thread to avoid Colab's async event loop conflict
result_container = {}

def run_crew():
    result_container['result'] = crew.kickoff()

thread = threading.Thread(target=run_crew)
print("🚀 Starting Crew run...")
thread.start()
thread.join(timeout=300)  # 5 minute timeout

if 'result' in result_container:
    result = result_container['result']
    print("\n✅ Crew run complete")
    print(result)
else:
    print("❌ Crew timed out or failed")

🚀 Starting Crew run...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f2741f75-9b10-46b8-8c81-5bb21cd50d62                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Fetch RSS feeds from these 3 URLs:                                                                         │
│      - https://techcrunch.com/feed/                                                                             │
│      - https://techcrunch.com/category/startups/feed/                                                           │
│      - https://venturebeat.com/feed/                                                                            │
│                                                                                                                 │
│      Then:                                                                                                      │
│      1. Filter entries where the title contains funding keywords:                                               │
│         'raises', 'funding', 'million', 'series', 'seed', 'venture', 'backed', 'investment'                     │
│      2. Filter entries published in the last 7 days only                                                        │
│      3. Deduplicate on title                                                                                    │
│      4. Exclude entries with noise phrases: 'layoffs', 'acqui', 'IPO', 'public offering', 'bankrupt'            │
│      5. Return a list of entries, each with: title, summary, published date, and link                           │
│                                                                                                                 │
│      Use the feedparser library to fetch the feeds.                                                             │
│                                                                                                                 │
│  ID: 6f0773e8-e571-4121-9b81-0d3f127852ce                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Startup Signal Scout                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Fetch RSS feeds from these 3 URLs:                                                                         │
│      - https://techcrunch.com/feed/                                                                             │
│      - https://techcrunch.com/category/startups/feed/                                                           │
│      - https://venturebeat.com/feed/                                                                            │
│                                                                                                                 │
│      Then:                                                                                                      │
│      1. Filter entries where the title contains funding keywords:                                               │
│         'raises', 'funding', 'million', 'series', 'seed', 'venture', 'backed', 'investment'                     │
│      2. Filter entries published in the last 7 days only                                                        │
│      3. Deduplicate on title                                                                                    │
│      4. Exclude entries with noise phrases: 'layoffs', 'acqui', 'IPO', 'public offering', 'bankrupt'            │
│      5. Return a list of entries, each with: title, summary, published date, and link                           │
│                                                                                                                 │
│      Use the feedparser library to fetch the feeds.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: PM Fit Analyst                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  [                                                                                                              │
│    {                                                                                                            │
│      "company": "AI-powered mental health startup",                                                             │
│      "amount": "$15M",                                                                                          │
│      "stage": "Series A",                                                                                       │
│      "sector": "Mental health",                                                                                 │
│      "fit_score": 8,                                                                                            │
│      "reason": "Strong AI fit, although not a perfect B2B SaaS or marketplace fit, still relevant due to AI     │
│  and Series A stage",                                                                                           │
│      "action": "reach out now",                                                                                 │
│      "key_people": "Not found"                                                                                  │
│    },                                                                                                           │
│    {                                                                                                            │
│      "company": "New York-based fintech startup",                                                               │
│      "amount": "$20M",                                                                                          │
│      "stage": "Series B",                                                                                       │
│      "sector": "Fintech",                                                                                       │
│      "fit_score": 7,                                                                                            │
│      "reason": "B2B fintech fit, relevant due to Series B stage, although not a perfect SaaS or AI match",      │
│      "action": "monitor",                                                                                       │
│      "key_people": "Not found"                                                                                  │
│    },                                                                                                           │
│    {                                                                                                            │
│      "company": "Virtual event platform",                                                                       │
│      "amount": "$10M",                                                                                          │
│      "stage": "Seed",                                                                                           │
│      "sector": "Virtual event",                                                                                 │
│      "fit_score": 4,                                                                                            │
│      "reason": "Not a strong B2B SaaS, AI, or marketplace fit, and it's a Seed round, which is less             │
│  favorable",                                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      You will receive a list of structured startup entries from the Researcher.                                 │
│                                                                                                                 │
│      Score each startup for PM fit on a scale of 1-10 based on this candidate profile:                          │
│      - 7+ years experience across Amazon, Deloitte, TCS                                                         │
│      - Strong in B2B SaaS, marketplace, and AI-powered products                                                 │
│      - MBA from UC Davis, graduating 2026                                                                       │
│      - Targeting Series A and Series B startups                                                                 │
│      - Open to well-funded Seed rounds                                                                          │
│      - Low interest in Hardware and pure consumer social                                                        │
│                                                                                                                 │
│      For each startup provide:                                                                                  │
│      - company: copy the EXACT company name from Researcher output (never use 'Unknown' or placeholder names)   │
│      - amount: copy exactly from Researcher                                                                     │
│      - stage: copy exactly from Researcher                                                                      │
│      - sector: copy exactly from Researcher                                                                     │
│      - fit_score: integer 1-10                                                                                  │
│      - reason: one sentence explaining the score                                                                │
│      - action: one of 'reach out now' (8-10), 'monitor' (5-7), 'skip' (1-4)                                     │
│      - key_people: founder/CEO/CTO names if mentioned in title or summary, otherwise 'Not found'                │
│                                                                                                                 │
│      Return ONLY a valid JSON array. No explanation, no markdown, no code blocks.                               │
│      Sort by fit_score descending.                                                                              │
│      Example:                                                                                                   │
│      [{"company": "Status AI", "amount": "$17M", "stage": "Series A", "sector": "AI", "fit_score": 9,           │
│  "reason": "Strong AI B2B fit", "action": "reach out now", "key_people": "Jane Smith (CEO)"}]                   │
│                                                                                                                 │
│  Agent: PM Fit Analyst                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ Crew run complete
[
  {
    "company": "AI-powered mental health startup",
    "amount": "$15M",
    "stage": "Series A",
    "sector": "Mental health",
    "fit_score": 8,
    "reason": "Strong AI fit, although not a perfect B2B SaaS or marketplace fit, still relevant due to AI and Series A stage",
    "action": "reach out now",
    "key_people": "Not found"
  },
  {
    "company": "New York-based fintech startup",
    "amount": "$20M",
    "stage": "Series B",
    "sector": "Fintech",
    "fit_score": 7,
    "reason": "B2B fintech fit, relevant due to Series B stage, although not a perfect SaaS or AI match",
    "action": "monitor",
    "key_people": "Not found"
  },
  {
    "company": "Virtual event platform",
    "amount": "$10M",
    "stage": "Seed",
    "sector": "Virtual event",
    "fit_score": 4,
    "reason": "Not a strong B2B SaaS, AI, or marketplace fit, and it's a Seed round, which is less favorable",
    "action": "skip",
    "key_people": "Not found"
  }
]


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f2741f75-9b10-46b8-8c81-5bb21cd50d62                                                                       │
│  Final Output: [                                                                                                │
│    {                                                                                                            │
│      "company": "AI-powered mental health startup",                                                             │
│      "amount": "$15M",                                                                                          │
│      "stage": "Series A",                                                                                       │
│      "sector": "Mental health",                                                                                 │
│      "fit_score": 8,                                                                                            │
│      "reason": "Strong AI fit, although not a perfect B2B SaaS or marketplace fit, still relevant due to AI     │
│  and Series A stage",                                                                                           │
│      "action": "reach out now",                                                                                 │
│      "key_people": "Not found"                                                                                  │
│    },                                                                                                           │
│    {                                                                                                            │
│      "company": "New York-based fintech startup",                                                               │
│      "amount": "$20M",                                                                                          │
│      "stage": "Series B",                                                                                       │
│      "sector": "Fintech",                                                                                       │
│      "fit_score": 7,                                                                                            │
│      "reason": "B2B fintech fit, relevant due to Series B stage, although not a perfect SaaS or AI match",      │
│      "action": "monitor",                                                                                       │
│      "key_people": "Not found"                                                                                  │
│    },                                                                                                           │
│    {                                                                                                            │
│      "company": "Virtual event platform",                                                                       │
│      "amount": "$10M",                                                                                          │
│      "stage": "Seed",                                                                                           │
│      "sector": "Virtual event",                                                                                 │
│      "fit_score": 4,                                                                                            │
│      "reason": "Not a strong B2B SaaS, AI, or marketplace fit, and it's a Seed round, which is less             │
│  favorable",                                          

In [ ]:
# Step 5: Parse Crew output into a ranked dataframe
import pandas as pd
import json
import re

# Extract the raw string from the CrewOutput object
raw = str(result)

# Find the JSON array in the output
match = re.search(r'\[.*\]', raw, re.DOTALL)

if match:
    json_str = match.group(0)
    data = json.loads(json_str)
    df_final = pd.DataFrame(data)
    df_final = df_final.sort_values('fit_score', ascending=False).reset_index(drop=True)
    print("✅ Final ranked startup list:")
    print(df_final[['company', 'amount', 'stage', 'sector', 'fit_score', 'action', 'reason']].to_string(index=False))
else:
    print("⚠️ Could not parse JSON from result. Raw output:")
    print(raw)

✅ Final ranked startup list:
                         company amount    stage        sector  fit_score        action                                                                                                         reason
AI-powered mental health startup   $15M Series A Mental health          8 reach out now Strong AI fit, although not a perfect B2B SaaS or marketplace fit, still relevant due to AI and Series A stage
  New York-based fintech startup   $20M Series B       Fintech          7       monitor                       B2B fintech fit, relevant due to Series B stage, although not a perfect SaaS or AI match
          Virtual event platform   $10M     Seed Virtual event          4          skip                  Not a strong B2B SaaS, AI, or marketplace fit, and it's a Seed round, which is less favorable


In [ ]:
#writing data to google sheet

In [ ]:
#sheet = client.open("Startup Signal Tracker").sheet1

In [ ]:
# Step 6: Export to Google Sheets (with timestamp + historical logging)
import gspread
from google.oauth2.service_account import Credentials
from google.colab import files
from datetime import datetime
import json

# Upload your JSON key file
print("Upload your service account JSON key file:")
uploaded = files.upload()

# Load credentials
key_filename = list(uploaded.keys())[0]
key_data = json.loads(uploaded[key_filename])

scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_info(key_data, scopes=scopes)
client = gspread.authorize(creds)

# Open the sheet
sheet = client.open("Startup Signal Tracker").sheet1

# Write headers only on first run, append on subsequent runs
headers = ['run_timestamp', 'company', 'amount', 'stage', 'sector', 'fit_score', 'action', 'reason']
existing = sheet.get_all_values()
if len(existing) == 0:
    sheet.append_row(headers)

# Timestamp for this run
run_time = datetime.now().strftime("%Y-%m-%d %H:%M")

# Append rows
for _, row in df_final.iterrows():
    sheet.append_row([
        run_time,
        row['company'],
        row['amount'],
        row['stage'],
        row['sector'],
        int(row['fit_score']),
        row['action'],
        row['reason']
    ])

print(f"✅ {len(df_final)} rows written to Google Sheets at {run_time}")

Upload your service account JSON key file:


Saving upbeat-medley-496904-i8-f2bdb8bedbbf.json to upbeat-medley-496904-i8-f2bdb8bedbbf.json
✅ 3 rows written to Google Sheets at 2026-05-23 20:59
